# INFS5705 AI for Business Analytics in Practice
## Week 3 Workshop Hands-On Activity 1 – Classifying Images using Supervised Learning in Python
### Let’s build a simplified AI-based Pizza Checker using Deep Learning in Python
We would like to check if the correct pizza is made, either a pepperoni pizza or a plain cheese pizza. To do that, we will train an AI model using a dataset of pre-classified images of pepperoni pizzas and plain cheese pizza. We will create, train, and evaluate a Convolutional Neural Network (CNN) model with Keras and TensorFlow for a binary image classification task.

#### Load the dependencies
The first step is to define the functions and classes (dependencies) to be used to build your machine learning model. Several libraries are imported for data manipulation, model creation, training, and evaluation:

* keras for building the neural network,
* sklearn.metrics for evaluation metrics,
* tensorflow for backend operations,
* cv2 (OpenCV) for image processing,
* os for file system operations,
* numpy for numerical operations,
* matplotlib.pyplot and seaborn for plotting, and
* shutil library to unpack archives.

In [ ]:
# Load the libraries needed
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D , MaxPool2D , Flatten , Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report,confusion_matrix
import tensorflow as tf
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shutil

#### Unzipping Training and Testing Data

As zip files are uploaded to your Colab Notebook, we need to unzip them using the following code. These zipped files contain the image data for training and testing the model, organised into folders named after the classes (Cheese and Pepperoni).

In [ ]:
# Unzip Training.zip and Testing.zip
shutil.unpack_archive("Training.zip", "/content/")
shutil.unpack_archive("Testing.zip", "/content/")

#### Data Preparation

We need to define the labels to be used in this model, i.e. 'Cheese' and 'Pepperoni', as well as the preferred image size to be used in the model (100 pixels by 100 pixels).

In [ ]:
# Define labels and image size
labels = ['Cheese', 'Pepperoni']
img_size = 100

We also define a function that reads the image files and pre-process them. This function loads images from a specified directory (data_dir), converts them from BGR to RGB (since OpenCV loads images in BGR format), resises them to 100x100 pixels, and appends them to a list along with their class labels. The data is then returned as a NumPy array.

In [ ]:
# Define the get_data function that loads the images
def get_data(data_dir):
    data = []
    for label in labels:
        path = os.path.join(data_dir, label)
        class_num = labels.index(label)
        for img in os.listdir(path):
            try:
                img_arr = cv2.imread(os.path.join(path, img))
                if img_arr is None:
                    continue  # Skip files that aren't valid images
                img_arr = cv2.cvtColor(img_arr, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB format
                resized_arr = cv2.resize(img_arr, (img_size, img_size))  # Reshaping images to preferred size
                data.append([resized_arr, class_num])
            except Exception as e:
                print(f"Error processing image {img}: {e}")
    # Convert data to a NumPy array for the image features, and a list for the labels
    X = np.array([item[0] for item in data], dtype=object)
    y = np.array([item[1] for item in data])
    return X, y

#### Loading and Preparing Data

Next, we load the training and testing data by calling the get_data function defined earlier, then prepare it for the training process. The datasets are split into features (x_train, x_test) and labels (y_train, y_test), and the feature data is normalised by dividing by 255 (to bring pixel values into the range [0,1]). The labels are also converted to NumPy arrays.

In [ ]:
# Load the training images
X_train, y_train = get_data('/content/Training')
# Load the testing images
X_test, y_test = get_data('/content/Testing')

# Normalise the data
X_train = np.array(X_train, dtype=np.float32) / 255
X_test = np.array(X_test, dtype=np.float32) / 255

# The reshaping of X_train and X_test to ensure they have the correct shape should be handled directly (assuming they are already in a 4D array format needed for CNNs):
X_train = X_train.reshape(-1, img_size, img_size, 3)
X_test = X_test.reshape(-1, img_size, img_size, 3)

# Ensure y_train and y_test are numpy arrays (they should already be, but this is just to be safe)
y_train = np.array(y_train, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)

#### Model Creation
A sequential CNN model is created with:

* Three convolutional layers, each followed by a max-pooling layer. The first layer specifies the input shape as 100x100 pixels with 3 color channels (RGB).
* A dropout layer to reduce overfitting.
* A flatten layer to convert the 2D feature maps into a 1D vector.
* Two dense (fully connected) layers, the final one using a softmax activation function for binary classification.

In [ ]:
# Initialise a Convolutional Neural Network (CNN) model
model = Sequential()
model.add(Conv2D(32,3,padding="same", activation="relu", input_shape=(img_size,img_size,3)))
model.add(MaxPool2D())

model.add(Conv2D(32, 3, padding="same", activation="relu"))
model.add(MaxPool2D())

model.add(Conv2D(64, 3, padding="same", activation="relu"))
model.add(MaxPool2D())
model.add(Dropout(0.4))

model.add(Flatten())
model.add(Dense(128,activation="relu"))
model.add(Dense(2, activation="softmax"))

model.summary()

#### Model Compilation

The model is compiled with the Adam optimiser and the sparse categorical crossentropy loss function. This setup is typical for multi-class classification tasks, even though this is a binary classification, represented as a 2-class problem.

In [ ]:
# Compile the model
opt = Adam()
model.compile(optimizer = opt , loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) , metrics = ['accuracy'])

### Model Training

The model is trained on the training data with validation on the testing data for 10 epochs. Training history is plotted to visualise the accuracy over epochs for both training and validation data.

In [ ]:
# Fit the model to the data
history = model.fit(X_train,y_train,epochs = 10 , validation_data = (X_test, y_test))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

#### Model Evaluation

Finally, the model's performance is evaluated on the test set. Predictions are made using the model.predict method, and the results are analysed using a confusion matrix and a classification report, which provides precision, recall, f1-score for each class.

In [ ]:
# Classify images (predict) using the Convolutional Neural Network (CNN) model
predictions = model.predict(X_test)
predictions = np.argmax(predictions, axis=1)
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions, target_names = ['Cheese (Class 0)','Pepperoni (Class 1)']))